<a href="https://colab.research.google.com/github/hmmnyamminji/DL/blob/main/day17_practice2_%EC%85%80%ED%94%84%EC%96%B4%ED%85%90%EC%85%98_%ED%95%B4%EB%B6%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 셀 1. Q · K · V는 어디서 오나 - '같은 X'의 세 가지 변신
# 셀프(self) 어텐션: 질문도, 색인도, 내용도 전부 '같은 문장 X'에서 만든다. X = '영화 정말 재미있다'
# Q = X @ W_q / K = X @ W_k / V = X @ W_v
# W_q, W_k, W_v는 학습되는 행렬 (nn.Linear)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, time
import matplotlib.pyplot as plt

torch.manual_seed(42)

In [ ]:
class SelfAttention(nn.Module):
  def __init__(self, dim):
    super().__init__()
    self.W_q = nn.Linear(dim, dim, bias=False) # Q · K · V를 만드는 학습 행렬 3개
    self.W_k = nn.Linear(dim, dim, bias=False)
    self.W_v = nn.Linear(dim, dim, bias=False)
    self.scale = math.sqrt(dim) # √d : 점수 나눈 값

  def forward(self, x): # x1 (B, 단어수, dim)
    Q, K, V = self.W_q(x), self.W_k(x), self.W_v(x)
    print(Q, K, V)
    attn = F.softmax(Q @ K.transpose(1,2) / self.scale, dim=-1) # K.transpose(1,2): 전치 1번 · 2번 축을 맞바꿈 (B,단어,dim)→(B,dim,단어)
    self.attn_map = attn # 시각화용 보관
    return attn @ V # 관련도 무게 x 내용

In [ ]:
attn_layer = SelfAttention(dim=16)

x = torch.rand(1, 3, 16) # 단어 3개짜리 문장

out = attn_layer(x)

print("입력:", tuple(x.shape), "→출력:", tuple(out.shape), "(모양 유지)")

tensor([[[ 0.0664, -0.0192, -0.0445, -0.1421,  0.2908,  0.2692, -0.1573,
          -0.2889, -0.4219, -0.1243, -0.2879, -0.0334,  0.0776, -0.7016,
           0.1852, -0.2336],
         [-0.0188,  0.1943,  0.1072, -0.1108,  0.0513, -0.0049, -0.6249,
          -0.5416, -0.6848,  0.0315, -0.1768, -0.2715, -0.0810, -0.7779,
           0.2367, -0.5470],
         [-0.1728,  0.1683,  0.2826, -0.6355,  0.0044,  0.1285, -0.2556,
          -0.6366, -0.6933,  0.1107, -0.0612, -0.2643, -0.3244, -0.4693,
           0.3019, -0.3812]]], grad_fn=<UnsafeViewBackward0>) tensor([[[-0.0814,  0.6482, -0.3054,  0.3760,  0.0719, -0.0381,  0.4603,
           0.0643,  0.0917,  0.3635,  0.4602, -0.3358,  0.0653, -0.2504,
           0.2478,  0.4148],
         [-0.2380,  0.9267, -0.6161,  0.4497,  0.0382, -0.1492,  0.5878,
           0.4378, -0.0286,  0.2944,  0.4874, -0.5813,  0.0738, -0.3996,
           0.4079,  0.5041],
         [-0.4864,  0.7481, -0.1614,  0.0927,  0.2689, -0.2311,  0.6168,
           0.3378, 

In [ ]:
# 첫 단어의 기울기가 산다
def first_word_grad(layer, seq_len, dim=32): #RNN / LSTM / 어텐션
  torch.manual_seed(0)
  x = torch.rand(1, seq_len, dim, requires_grad=True)
  out = layer(x)
  if isinstance(out, tuple): out = out[0] # RNN/LSTM은 (out, h) 튜플, 어텐션은 텐서 하나
  out[0, -1].sum().backward() # 0이면 문장의 마지막 단어 출력에서 역전파
  return x.grad[0,0].abs().mean().item() # 0번 문장 '첫 단어' 자리의 기울기

print("\n[첫 단어의 기울기 - 문장길이 100]")

rnn = nn.RNN(32, 32, batch_first=True)
lstm = nn.LSTM(32, 32, batch_first=True)
attn = SelfAttention(32)

print(f"RNN: {first_word_grad(rnn, 100):.8f}")
print(f"LSTM: {first_word_grad(lstm, 100):.8f}")
print(f"Attention: {first_word_grad(attn, 100):.8f}")


[첫 단어의 기울기 - 문장길이 100]
RNN: 0.00000000
LSTM: 0.00000000
tensor([[[ 0.1542, -0.2684,  0.6600,  ..., -0.0455, -0.3847,  0.1946],
         [-0.0419, -0.4798,  0.7989,  ..., -0.3050, -0.2502,  0.0925],
         [ 0.2335, -0.2530,  0.8283,  ..., -0.0671, -0.2875,  0.3820],
         ...,
         [ 0.2918, -0.3717,  0.6620,  ..., -0.3564, -0.1343,  0.4237],
         [ 0.4473, -0.5506,  0.8187,  ..., -0.2184, -0.2150, -0.0302],
         [ 0.1108, -0.0912,  0.4751,  ..., -0.2747, -0.3532,  0.0688]]],
       grad_fn=<UnsafeViewBackward0>) tensor([[[-0.2855, -0.4629,  0.0089,  ..., -0.1252,  0.1176, -0.0177],
         [-0.2144, -0.1751, -0.1405,  ..., -0.1070, -0.2628, -0.0057],
         [ 0.3135, -0.5059, -0.0280,  ..., -0.0857,  0.1288,  0.2770],
         ...,
         [ 0.1103, -0.3947,  0.3522,  ..., -0.4400,  0.2146,  0.3026],
         [ 0.0166, -0.5921, -0.0752,  ..., -0.0580, -0.1273,  0.2434],
         [-0.1633, -0.4069, -0.1284,  ..., -0.0954, -0.1469,  0.2810]]],
       grad_fn=<Unsaf